In [96]:
import os
import pandas as pd
import numpy as np
import rasterio
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from google.colab import drive

drive.mount('/content/drive')

# Paths to datasets
DATA_PATHS = {
    "FAO": "/content/drive/My Drive/FarmWise/Cleaned_Data/FAO/",
    "SoilGrids250m": "/content/drive/My Drive/FarmWise/Cleaned_Data/SoilGrids250m/",
    "Agridata": "/content/drive/My Drive/FarmWise/Cleaned_Data/agridata/"
}

# Initialize storage for processed data
processed_data = {}

# ---------------------- FAO DATA PROCESSING ----------------------
fao_files = []
for root, _, files in os.walk(DATA_PATHS['FAO']):
    for file in files:
        if file.endswith('.csv'):
            fao_files.append(os.path.join(root, file))
fao_dataframes = {}

for file in fao_files:
    file_path = os.path.join(DATA_PATHS["FAO"], file)
    df = pd.read_csv(file_path)

    # Drop duplicates
    df.drop_duplicates(inplace=True)

    # Identify numeric columns
    num_cols = df.select_dtypes(include=['float64', 'int64']).columns

    # Handle missing values only for numeric columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].mean())

    # Normalize numerical features
    scaler = MinMaxScaler()
    if not df[num_cols].empty:
        df[num_cols] = scaler.fit_transform(df[num_cols])

    # Feature engineering - Adding interaction terms (if needed)
    if 'Year' in df.columns:
        df['Year_Squared'] = df['Year'] ** 2

    fao_dataframes[file] = df

processed_data['FAO'] = fao_dataframes
print("FAO Data Processed ✔")

# ---------------------- SOILGRIDS250m DATA PROCESSING ----------------------
soil_files = []
for root, _, files in os.walk(DATA_PATHS['SoilGrids250m']):
    for file in files:
        if file.endswith('.csv'):
            soil_files.append(os.path.join(root, file))

soil_dataframes = {}

for file in soil_files:
    file_path = os.path.join(DATA_PATHS["SoilGrids250m"], file)
    df = pd.read_csv(file_path)

    # Drop duplicates
    df.drop_duplicates(inplace=True)

    # Identify numeric columns
    num_cols = df.select_dtypes(include=['float64', 'int64']).columns

    # Handle missing values only for numeric columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].mean())

    # Normalize numerical features
    scaler = MinMaxScaler()
    if not df[num_cols].empty:
        df[num_cols] = scaler.fit_transform(df[num_cols])

    # Feature engineering - Adding interaction terms (if needed)
    if 'Year' in df.columns:
        df['Year_Squared'] = df['Year'] ** 2

    soil_dataframes[file] = df

processed_data['SoilGrids250m'] = soil_dataframes
print("SoilGrids250m Data Processed ✔")


# ---------------------- AGRIDATA DATA PROCESSING ----------------------
agridata_files = []
for root, _, files in os.walk(DATA_PATHS['Agridata']):
    for file in files:
        if file.endswith('.csv'):
            agridata_files.append(os.path.join(root, file))
agridata_dataframes = {}

for file in agridata_files:
    file_path = os.path.join(DATA_PATHS["Agridata"], file)
    df = pd.read_csv(file_path)

    # Identify numeric columns
    num_cols = df.select_dtypes(include=['float64', 'int64']).columns

    # Handle missing values only for numeric columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].mean())

    # Fill categorical missing values using forward fill
    df.ffill(inplace=True)
    df.bfill(inplace=True)

    # Standardize numerical data
    if len(num_cols) > 0:
        scaler = StandardScaler()
        if not df[num_cols].empty:
            df[num_cols] = scaler.fit_transform(df[num_cols])

    # Encode categorical variables (if any)
    cat_cols = df.select_dtypes(include=['object']).columns
    if len(cat_cols) > 0:
        encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        encoded_df = pd.DataFrame(encoder.fit_transform(df[cat_cols]))
        df = df.drop(columns=cat_cols).reset_index(drop=True)
    if not encoded_df.empty and not df.empty:
        df = pd.concat([df.reset_index(drop=True), encoded_df.reset_index(drop=True)], axis=1)



    # Apply PCA for dimensionality reduction (if necessary)
    if df.shape[1] > 10:  # Apply PCA only if we have many features
        df.columns = df.columns.astype(str)
        pca = PCA(n_components=min(10, min(df.shape[0], df.shape[1])))
        df_pca = pca.fit_transform(df.fillna(0))

        df = pd.DataFrame(df_pca, columns=[f'PC{i+1}' for i in range(df_pca.shape[1])])

    agridata_dataframes[file] = df

processed_data['Agridata'] = agridata_dataframes
print("Agridata Data Processed ✔")

# ---------------------- DATA VALIDATION & VISUALIZATION ----------------------
for category, data in processed_data.items():
    print(f"\n--- {category} Data Overview ---")
    if isinstance(data, dict):
        for file, df in data.items():
           print(f"File: {file}")

           if df.empty:  # Skip empty DataFrames
               print("⚠ Warning: DataFrame is empty, skipping this file.\n")
        continue

    print(df.describe())
    print("\n")


           # Plot distributions
    num_cols = df.select_dtypes(include=['float64', 'int64']).columns
    if len(num_cols) > 0:
        df[num_cols].hist(figsize=(10, 5))
        plt.suptitle(f"{file} - Numerical Features Distribution")
        plt.show()



print("All datasets processed, validated, and saved!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FAO Data Processed ✔
SoilGrids250m Data Processed ✔
Agridata Data Processed ✔

--- FAO Data Overview ---
File: /content/drive/My Drive/FarmWise/Cleaned_Data/FAO/Production/Production Indices.csv
File: /content/drive/My Drive/FarmWise/Cleaned_Data/FAO/Production/Crops and livestock products.csv
File: /content/drive/My Drive/FarmWise/Cleaned_Data/FAO/Production/Value of Agricultural Production.csv
File: /content/drive/My Drive/FarmWise/Cleaned_Data/FAO/Climate Change-Agrifood systems emissions/Land use and change/Emissions from Drained organic soils.csv
File: /content/drive/My Drive/FarmWise/Cleaned_Data/FAO/Climate Change-Agrifood systems emissions/Totals and Indicators/Emissions totals.csv
File: /content/drive/My Drive/FarmWise/Cleaned_Data/FAO/Climate Change-Agrifood systems emissions/Totals and Indicators/Emissions indicators.csv
File: /content/drive/My Dri

In [97]:
import os

output_folder = "/content/drive/My Drive/FarmWise/Processed_Data"

# Ensure the output folder exists
os.makedirs(output_folder, exist_ok=True)

for category, data in processed_data.items():
    category_folder = os.path.join(output_folder, category)
    os.makedirs(category_folder, exist_ok=True)  # Create subfolder

    for file, df in data.items():
        output_filename = os.path.basename(file)
        output_path = os.path.join(category_folder, output_filename)
        df.to_csv(output_path, index=False)
        print(f"✅ Saved: {output_path}")


✅ Saved: /content/drive/My Drive/FarmWise/Processed_Data/FAO/Production Indices.csv
✅ Saved: /content/drive/My Drive/FarmWise/Processed_Data/FAO/Crops and livestock products.csv
✅ Saved: /content/drive/My Drive/FarmWise/Processed_Data/FAO/Value of Agricultural Production.csv
✅ Saved: /content/drive/My Drive/FarmWise/Processed_Data/FAO/Emissions from Drained organic soils.csv
✅ Saved: /content/drive/My Drive/FarmWise/Processed_Data/FAO/Emissions totals.csv
✅ Saved: /content/drive/My Drive/FarmWise/Processed_Data/FAO/Emissions indicators.csv
✅ Saved: /content/drive/My Drive/FarmWise/Processed_Data/FAO/Emissions intensities.csv
✅ Saved: /content/drive/My Drive/FarmWise/Processed_Data/FAO/Emissions from Crops.csv
✅ Saved: /content/drive/My Drive/FarmWise/Processed_Data/FAO/Emissions from Livestock.csv
✅ Saved: /content/drive/My Drive/FarmWise/Processed_Data/FAO/Emissions from Energy use in agriculture.csv
✅ Saved: /content/drive/My Drive/FarmWise/Processed_Data/FAO/Employment Indicators-Ag